
# NETs Detectron2 pipeline 




In [4]:

# Optional: mount Google Drive in Colab
from google.colab import drive
drive.mount('/content/gdrive')


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [5]:

# Basic imports
import os
import json
import random
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt


In [ ]:

# Install Detectron2 in Colab

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

!pip -q install 'git+https://github.com/facebookresearch/detectron2.git'


Torch version: 2.10.0+cpu
CUDA available: False
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 27.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━ 112.6/154.5 kB 24.2 kB/s eta 0:00:02
ERROR: Operation cancelled by user


In [ ]:

# Detectron2 imports
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.evaluation import COCOEvaluator
from detectron2.data import build_detection_test_loader
from detectron2.engine import hooks



## 1) Sanitize the COCO annotations

Your original notebook showed a key clue during evaluation:

- the validation loader reported **3 categories**
- one of them had **0 instances**
- but your model was configured with `NUM_CLASSES = 2`

That mismatch can cause confusing label mapping and make it *look* like the model is predicting only one class.

The function below:
- removes unused categories
- keeps only categories that actually appear in annotations
- remaps category ids to **1..N** (clean COCO format)


In [ ]:

def sanitize_coco_json(input_json_path, output_json_path):
    with open(input_json_path, "r") as f:
        coco = json.load(f)

    annotations = coco["annotations"]
    categories = coco["categories"]

    used_ids = sorted({ann["category_id"] for ann in annotations})
    used_categories = [cat for cat in categories if cat["id"] in used_ids]

    if not used_categories:
        raise ValueError("No used categories were found in the annotations.")

    old_to_new = {}
    new_categories = []
    for new_id, cat in enumerate(sorted(used_categories, key=lambda x: x["id"]), start=1):
        old_to_new[cat["id"]] = new_id
        new_cat = dict(cat)
        new_cat["id"] = new_id
        new_categories.append(new_cat)

    for ann in annotations:
        ann["category_id"] = old_to_new[ann["category_id"]]

    coco["categories"] = new_categories

    with open(output_json_path, "w") as f:
        json.dump(coco, f)

    print(f"Saved sanitized JSON to: {output_json_path}")
    print("Category mapping:", old_to_new)
    print("Final categories:", [(c["id"], c["name"]) for c in new_categories])


In [ ]:

# Update these paths
TRAIN_JSON = "/content/gdrive/MyDrive/Capstone_folder/train_dataset/_annotations.coco.json"
VAL_JSON   = "/content/gdrive/MyDrive/Capstone_folder/validate/_annotations.coco.json"

TRAIN_IMG_DIR = "/content/gdrive/MyDrive/Capstone_folder/train_dataset"
VAL_IMG_DIR   = "/content/gdrive/MyDrive/Capstone_folder/validate"

TRAIN_JSON_CLEAN = "/content/gdrive/MyDrive/Capstone_folder/train_dataset/_annotations_clean.coco.json"
VAL_JSON_CLEAN   = "/content/gdrive/MyDrive/Capstone_folder/validate/_annotations_clean.coco.json"

sanitize_coco_json(TRAIN_JSON, TRAIN_JSON_CLEAN)
sanitize_coco_json(VAL_JSON, VAL_JSON_CLEAN)


## 2) Register the datasets

In [ ]:

# Clean up old registrations if they already exist
for name in ["nets_train", "nets_val"]:
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
    if name in MetadataCatalog.list():
        MetadataCatalog.remove(name)

register_coco_instances("nets_train", {}, TRAIN_JSON_CLEAN, TRAIN_IMG_DIR)
register_coco_instances("nets_val", {}, VAL_JSON_CLEAN, VAL_IMG_DIR)

train_dicts = DatasetCatalog.get("nets_train")
val_dicts = DatasetCatalog.get("nets_val")

train_meta = MetadataCatalog.get("nets_train")
val_meta = MetadataCatalog.get("nets_val")

print("Train size:", len(train_dicts))
print("Val size:", len(val_dicts))
print("thing_classes:", train_meta.thing_classes)


## 3) Inspect class distribution before training

In [ ]:

def summarize_dataset(dataset_dicts, metadata, name="dataset"):
    counter = Counter()
    images_per_class = defaultdict(int)

    for record in dataset_dicts:
        seen_in_image = set()
        for ann in record["annotations"]:
            cid = ann["category_id"]
            counter[cid] += 1
            seen_in_image.add(cid)
        for cid in seen_in_image:
            images_per_class[cid] += 1

    print(f"Summary for {name}")
    print("-" * 40)
    for cid, class_name in enumerate(metadata.thing_classes):
        print(
            f"class_id={cid:>2} | class_name={class_name:<20} | "
            f"instances={counter[cid]:>5} | images={images_per_class[cid]:>4}"
        )

summarize_dataset(train_dicts, train_meta, "train")
summarize_dataset(val_dicts, val_meta, "val")


In [ ]:

# Visual sanity check
sample = random.choice(train_dicts)
img = cv2.imread(sample["file_name"])
vis = Visualizer(img[:, :, ::-1], metadata=train_meta, scale=0.7)
out = vis.draw_dataset_dict(sample)
plt.figure(figsize=(10, 10))
plt.imshow(out.get_image()[:, :, ::-1])
plt.axis("off")
plt.show()



## 4) Configure training

Key upgrades relative to the original notebook:
- validation is enabled through `cfg.DATASETS.TEST`
- evaluation is run periodically
- score threshold is lowered to `0.3` at inference so weaker class predictions are not hidden


In [ ]:

OUTPUT_DIR = "/content/gdrive/MyDrive/Capstone_folder/models/Detectron2_models_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))

cfg.OUTPUT_DIR = OUTPUT_DIR
cfg.DATASETS.TRAIN = ("nets_train",)
cfg.DATASETS.TEST = ("nets_val",)

cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 5000
cfg.SOLVER.STEPS = []
cfg.SOLVER.CHECKPOINT_PERIOD = 500

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 256
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2

cfg.TEST.EVAL_PERIOD = 500
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3


In [ ]:

class TrainerWithVal(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        output_folder = output_folder or os.path.join(cfg.OUTPUT_DIR, "inference")
        return COCOEvaluator(dataset_name, output_dir=output_folder)

trainer = TrainerWithVal(cfg)
trainer.resume_or_load(resume=False)
trainer.train()


## 5) Load the final model and run inference

In [ ]:

cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
predictor = DefaultPredictor(cfg)


In [ ]:

# Inspect a validation image
record = random.choice(val_dicts)
im = cv2.imread(record["file_name"])
outputs = predictor(im)

print("Predicted class ids:", outputs["instances"].pred_classes.to("cpu").numpy())

v = Visualizer(
    im[:, :, ::-1],
    metadata=val_meta,
    scale=0.7,
    instance_mode=ColorMode.IMAGE_BW,
)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

plt.figure(figsize=(10, 10))
plt.imshow(out.get_image()[:, :, ::-1])
plt.axis("off")
plt.show()


In [ ]:

# Count predictions by class over the validation set
pred_counter = Counter()

for record in val_dicts:
    im = cv2.imread(record["file_name"])
    outputs = predictor(im)
    classes = outputs["instances"].pred_classes.to("cpu").numpy().tolist()
    pred_counter.update(classes)

print("Prediction counts by class id:", pred_counter)
for cid, name in enumerate(val_meta.thing_classes):
    print(f"{cid}: {name} -> {pred_counter[cid]} predictions")



## 6) Cleaner batch inference

This replaces the original mixed Colab/Mac path code with a portable version.


In [ ]:

TEST_IMAGE_DIR = "/content/gdrive/MyDrive/Capstone_folder/test_data"
TEST_OUTPUT_DIR = "/content/gdrive/MyDrive/Capstone_folder/test_predictions"
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

for filename in sorted(os.listdir(TEST_IMAGE_DIR)):
    ext = os.path.splitext(filename)[1].lower()
    if ext not in valid_exts:
        continue

    image_path = os.path.join(TEST_IMAGE_DIR, filename)
    image = cv2.imread(image_path)
    if image is None:
        print("Skipping unreadable image:", image_path)
        continue

    outputs = predictor(image)
    v = Visualizer(image[:, :, ::-1], metadata=train_meta, scale=0.7)
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

    save_path = os.path.join(TEST_OUTPUT_DIR, os.path.splitext(filename)[0] + "_result.png")
    cv2.imwrite(save_path, out.get_image()[:, :, ::-1])

print("Saved predictions to:", TEST_OUTPUT_DIR)



## 7) Notes

If the model still seems to predict mostly one class, check these in order:

1. The sanitized train/val JSONs both show exactly **2 classes**.
2. `train_meta.thing_classes` contains exactly the two expected names.
3. The validation dataset summary shows non-zero instances for both classes.
4. You are looking at predictions with `SCORE_THRESH_TEST = 0.3`, not `0.5`.
5. The labels in the raw annotations are visually correct.
